# 11 Gene-Level Feature Importance from PC Models

**Goal:** Trace PC-based model predictions back to individual genes.  
For qualifying tissue × pathology pairs (PC+Conf AUC ≥ 0.65, Δ ≥ 0.05).

**Two-step normalization** ensures gene importances are proper proportions (sum to 1):
1. **Step 1:** Renormalize RF Gini importances (PCs only) → `Σ RF_imp_norm = 1`
2. **Step 2:** Normalize |loadings| per PC → `Σ |loading_norm(i,j)| = 1` for each PC

**Formula:** `gene_j = Σ_i RF_imp_norm_i × |loading_norm(i,j)|`

→ `Σ_j gene_j = 1` — each gene's importance is a proportion of total predictive signal.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score

from gtex_biomarkers.config import Config
from gtex_biomarkers.data import load_cache, build_confounder_matrix, variance_filter
from gtex_biomarkers.labels import assign_donor_labels
from gtex_biomarkers.models import make_rf_model, _auc_feature_selection

Config.ensure_dirs()

# ── Load data ──
X_wb, blood_subjid, _, df_meta_url, df_age = load_cache()
X_conf = build_confounder_matrix(df_age, blood_subjid)

# ── Variance filter — same 20K genes as NB07 and NB10 ──
X_wb_var, gene_var = variance_filter(X_wb)
print(f'Variance filter: {X_wb.shape[1]:,} → {X_wb_var.shape[1]:,} genes')

# ── Ensembl ID → gene symbol ──
gene_map_df = pd.read_csv(Config.EXPR_FILE, sep="\t", skiprows=2,
                           usecols=["Name", "Description"], nrows=60000)
gene_id_to_name = dict(zip(gene_map_df["Name"].astype(str),
                            gene_map_df["Description"].astype(str)))

# ── NB07 direct RF importance (for validation) ──
rf_fi = pd.read_csv(Config.TABLES_DIR / "rf_feature_importances.csv")

print(f"Expression: {X_wb_var.shape[0]} samples x {X_wb_var.shape[1]:,} genes")
print(f"Gene mapping: {len(gene_id_to_name):,}")
print(f"NB07 RF importance: {len(rf_fi)} rows")

## 11.1 — Filter Qualifying Models

From NB10: PC+Conf AUC ≥ 0.65 and Δ ≥ 0.05.

In [ ]:
AUC_THRESH = 0.65
DELTA_THRESH = 0.05
N_PCS = 800
TOP_K_PCS = 100
TOP_N_GENES = 30
TOP_N_VAL = 50

pc_results = pd.read_csv(Config.TABLES_DIR / "pc_auc_results.csv")

qualified = pc_results[
    (pc_results["auc_pc_conf"] >= AUC_THRESH) & (pc_results["delta_pc"] >= DELTA_THRESH)
].reset_index(drop=True)

print(f"Total pairs: {len(pc_results)}")
print(f"Qualifying (AUC >= {AUC_THRESH}, Δ >= {DELTA_THRESH}): {len(qualified)}")
print()
display(qualified[["tissue", "category", "auc_pc_conf", "delta_pc"]].sort_values("delta_pc", ascending=False))

## 11.2 — Collect CV Artifacts

Leak-free CV on variance-filtered 20K genes.  
Per fold, save: PCA loadings, selected PC indices, **normalized** RF importances (PCs only, renormed to sum to 1).

In [ ]:
def run_cv_with_artifacts(X, y, groups, X_conf_sub,
                          n_pcs=N_PCS, top_k_pcs=TOP_K_PCS, tag=""):
    """Leak-free CV that returns PCA loadings + normalized RF importances per fold."""
    cv = StratifiedGroupKFold(n_splits=Config.N_SPLITS, shuffle=True,
                              random_state=Config.SEED)
    fold_artifacts = []

    for fold, (tr, te) in enumerate(cv.split(X, y, groups=groups), 1):
        Xtr = X.iloc[tr]
        ytr = y.iloc[tr]

        scaler = StandardScaler()
        Xtr_scaled = scaler.fit_transform(Xtr)
        n_comp = min(n_pcs, Xtr_scaled.shape[0] - 1, Xtr_scaled.shape[1])
        pca = PCA(n_components=n_comp, random_state=Config.SEED)
        Xtr_pcs = pca.fit_transform(Xtr_scaled)

        pc_cols = [f"PC{i+1}" for i in range(n_comp)]
        Xtr_pcs = pd.DataFrame(Xtr_pcs, columns=pc_cols, index=Xtr.index)

        top_pcs = _auc_feature_selection(Xtr_pcs, ytr, top_k=min(top_k_pcs, n_comp))
        selected_idx = [int(c.replace("PC", "")) - 1 for c in top_pcs]

        Xtr_sel = pd.concat([Xtr_pcs[top_pcs], X_conf_sub.iloc[tr]], axis=1)

        model = make_rf_model()
        model.fit(Xtr_sel, ytr)

        # Step 1: Slice PC importances and renormalize to sum to 1
        n_pc_features = len(top_pcs)
        rf_imp_pcs = model.feature_importances_[:n_pc_features].copy()
        rf_imp_pcs = rf_imp_pcs / rf_imp_pcs.sum()

        fold_artifacts.append({
            "loadings": pca.components_,
            "selected_idx": selected_idx,
            "rf_imp_pcs": rf_imp_pcs,
        })

    print(f"  {tag}: collected artifacts from {len(fold_artifacts)} folds")
    return fold_artifacts

In [ ]:
all_artifacts = {}

for i, row in qualified.iterrows():
    tissue, category = row["tissue"], row["category"]
    tag = f"{tissue} | {category}"
    print(f"\n[{i+1}/{len(qualified)}] {tag}")

    y, _, n_pos, n_neg = assign_donor_labels(
        df_meta_url, tissue, category, blood_subjid
    )
    keep = y.notna()
    y_clean = y[keep].astype(int)
    groups = blood_subjid[keep].astype(str)
    X_sub = X_wb_var.loc[keep]
    X_conf_sub = X_conf.loc[keep]

    print(f"  Samples: {n_pos} pos / {n_neg} neg (total {keep.sum()})")

    artifacts = run_cv_with_artifacts(X_sub, y_clean, groups, X_conf_sub, tag=tag)
    all_artifacts[(tissue, category)] = artifacts

    # Sanity: Step 1 normalization on fold 1
    print(f"  Fold 1 RF_imp sum (PCs): {artifacts[0]['rf_imp_pcs'].sum():.6f} (should be 1.0)")

print(f"\nDone — collected artifacts for {len(all_artifacts)} pairs.")

## 11.2b — Sanity Check: Both Normalizations

Verify on the first qualifying pair, fold 1:
1. **Step 1:** RF Gini importance (PCs only) sums to 1
2. **Step 2:** |loadings| per PC sum to 1 after normalization

In [ ]:
# Pick first pair, fold 1
first_key = list(all_artifacts.keys())[0]
art = all_artifacts[first_key][0]

print(f"Pair: {first_key[0]} | {first_key[1]}, Fold 1")
print()

# Step 1: RF importance (PCs only) sums to 1
rf_sum = art["rf_imp_pcs"].sum()
print(f"Step 1 — Σ RF_imp_norm (PCs only): {rf_sum:.6f}  (should be 1.0)")
print()

# Step 2: |loadings| per PC — before and after normalization
sel_loadings = art["loadings"][art["selected_idx"], :]
abs_loadings = np.abs(sel_loadings)
row_sums_before = abs_loadings.sum(axis=1)
abs_loadings_norm = abs_loadings / row_sums_before[:, None]
row_sums_after = abs_loadings_norm.sum(axis=1)

print(f"Step 2 — |loadings| per PC (showing first 5 of {len(art['selected_idx'])} PCs):")
print(f"  Before normalization — Σ|L|: {row_sums_before[:5].round(4).tolist()}")
print(f"  After normalization  — Σ|L|: {row_sums_after[:5].round(6).tolist()}")
print()

# End-to-end: gene importances should sum to 1
rf_imp = art["rf_imp_pcs"]
gene_imp_fold = (rf_imp[:, None] * abs_loadings_norm).sum(axis=0)
print(f"End-to-end — Σ gene_importance (single fold): {gene_imp_fold.sum():.6f}  (should be 1.0)")

## 11.3 — Compute Gene Importance (Normalized Absolute Loadings)

For each fold:
- Step 1 already done: `RF_imp_norm` sums to 1 across PCs
- Step 2: `|loading_norm(i,j)| = |loading(i,j)| / Σ_m |loading(i,m)|` — sums to 1 per PC

Average across 5 folds.

In [ ]:
gene_importance = {}

for (tissue, category), artifacts in all_artifacts.items():
    n_genes = X_wb_var.shape[1]
    imp = np.zeros(n_genes)

    for art in artifacts:
        sel_loadings = art["loadings"][art["selected_idx"], :]  # (top_k, n_genes)
        rf_imp = art["rf_imp_pcs"]  # already normalized to sum to 1

        # Step 2: normalize |loadings| per PC so each row sums to 1
        abs_loadings = np.abs(sel_loadings)
        row_sums = abs_loadings.sum(axis=1, keepdims=True)
        abs_loadings_norm = abs_loadings / row_sums

        imp += (rf_imp[:, None] * abs_loadings_norm).sum(axis=0)

    imp /= len(artifacts)
    gene_importance[(tissue, category)] = pd.Series(imp, index=X_wb_var.columns)

    top5 = [gene_id_to_name.get(g, g) for g in gene_importance[(tissue, category)].nlargest(5).index]
    gi_sum = imp.sum()
    print(f"{tissue} | {category} — sum={gi_sum:.4f} (should be ~1.0), top 5: {top5}")

## 11.4 — Top Genes per Qualifying Pair

In [ ]:
for (tissue, category), imp in gene_importance.items():
    top = imp.nlargest(TOP_N_GENES).sort_values()
    labels = [gene_id_to_name.get(g, g) for g in top.index]

    fig, ax = plt.subplots(figsize=(7, max(5, TOP_N_GENES * 0.25)))
    ax.barh(range(len(top)), top.values, color="#4878A8", height=0.7)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_xlabel("Gene importance (normalized)")
    ax.set_title(f"{tissue} | {category} \u2014 Top {TOP_N_GENES} Genes", fontsize=10)
    fig.tight_layout()
    plt.show()

## 11.5 — Validation vs Direct RF (NB07)

Sanity check against NB07's direct RF importance on raw genes.  
Low overlap is expected — PC-derived importance finds genes in coordinated patterns,  
direct RF finds individually discriminating genes.

In [ ]:
val_rows = []

for (tissue, category), imp in gene_importance.items():
    rf_pair = rf_fi[(rf_fi["tissue"] == tissue) & (rf_fi["category"] == category)]
    if rf_pair.empty:
        print(f"  {tissue} | {category}: no NB07 data — skipping")
        continue

    rf_direct = rf_pair.set_index("gene")["mean_importance"]
    common = imp.index.intersection(rf_direct.index)
    if len(common) < 10:
        continue

    rho, pval = spearmanr(imp[common], rf_direct[common])
    top_formula = set(imp.nlargest(TOP_N_VAL).index)
    top_rf = set(rf_direct.nlargest(TOP_N_VAL).index)
    jaccard = len(top_formula & top_rf) / len(top_formula | top_rf)

    val_rows.append({
        "tissue": tissue, "category": category,
        "n_overlap": len(common),
        "spearman": rho, "p_value": pval, "jaccard_top50": jaccard,
    })
    print(f"  {tissue} | {category} — rho={rho:.3f}, jaccard={jaccard:.3f}")

val_df = pd.DataFrame(val_rows)
if not val_df.empty:
    print(f"\nMean Spearman: {val_df['spearman'].mean():.3f}, "
          f"Mean Jaccard: {val_df['jaccard_top50'].mean():.3f}")
display(val_df)

## 11.6 — Export

In [ ]:
rows = []
for (tissue, category), imp in gene_importance.items():
    for gene in imp.nlargest(TOP_N_GENES).index:
        rows.append({
            "tissue": tissue, "category": category,
            "gene_id": gene,
            "gene_name": gene_id_to_name.get(gene, gene),
            "importance": imp[gene],
            "rank": imp.rank(ascending=False)[gene].astype(int),
        })

export_df = pd.DataFrame(rows).sort_values(["tissue", "category", "rank"]).reset_index(drop=True)
out_path = Config.TABLES_DIR / "pc_gene_importance.csv"
export_df.to_csv(out_path, index=False)
print(f"Saved {len(export_df)} rows to {out_path}")
display(export_df.head(10))

if not val_df.empty:
    val_df.to_csv(Config.TABLES_DIR / "validation_pc_vs_rf.csv", index=False)
    print(f"Saved validation to {Config.TABLES_DIR / 'validation_pc_vs_rf.csv'}")